# 난임 환자 임신 성공 여부 예측 — v4 최종본

## 핵심 변경사항
- **피처 ~65개로 축소** (147→~65, importance≥100 + 중복 제거)
- **신규 유의미 파생피처 12개** (임상 도메인 기반, 누수 없음)
- **Data Leakage 완전 차단**: train 통계만 medians dict로 전달
- **Optuna MedianPruner**: 저성능 trial 조기 종료
- **직접 가중 앙상블**: 스태킹 제거 → Optuna 200회 최적화
- **Permutation Importance 자동 피처 정제**
- **캐글/로컬 자동 감지**

# 난임 환자 임신 성공 여부 예측 — v4 최종본

## 핵심 변경사항
- **피처 65개로 축소** (147→65, Level-1 기준: importance≥100 + 중복 제거)
- **신규 유의미 파생피처 12개 추가** (누수 없음, train 통계 고정)
  - 임상 도메인 기반: 누적실패부담, 배아생존율, IVF집중도 등
  - 비선형 교호작용: 나이×배아품질, 시술부담×불임원인 등
- **Data Leakage 완전 차단**: 모든 통계값 train fit → medians dict로 test에 전달
- **Optuna MedianPruner**: 저성능 trial 조기 종료
- **직접 가중 앙상블**: 스태킹 제거 → Optuna 200회 가중치 최적화
- **캐글/로컬 자동 감지**

In [ ]:
import warnings, os, sys
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    subdirs  = os.listdir('/kaggle/input')
    DATA_DIR = f"/kaggle/input/{subdirs[0]}"
    OUT_DIR  = "/kaggle/working"
    print(f"✅ 캐글 환경 | {DATA_DIR} | {os.listdir(DATA_DIR)}")
else:
    DATA_DIR = "../data"
    OUT_DIR  = ".."
    print("✅ 로컬 환경")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance as pi_fn
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    import koreanize_matplotlib
except:
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

import torch
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE        = "cuda" if GPU_AVAILABLE else "cpu"
if GPU_AVAILABLE:
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}  "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
else:
    print("⚠️  CPU 모드")

XGB_DEVICE_PARAMS = {"device": "cuda"} if GPU_AVAILABLE else {}
LGB_DEVICE_PARAMS = {"device": "gpu", "gpu_platform_id": 0, "gpu_device_id": 0} if GPU_AVAILABLE else {}
CAT_DEVICE_PARAMS = {"task_type": "GPU", "devices": "0"} if GPU_AVAILABLE else {}

RANDOM_STATE = 42
N_SPLITS     = 5
N_OPTUNA     = 50
SEEDS        = [42, 2024, 777, 1234, 9999]
np.random.seed(RANDOM_STATE)

train = pd.read_csv(f'{DATA_DIR}/train.csv', encoding="utf-8-sig")
test  = pd.read_csv(f'{DATA_DIR}/test.csv',  encoding="utf-8-sig")
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv', encoding="utf-8-sig")
for df in [train, test, sub]:
    df.columns = [c.strip() for c in df.columns]

TARGET = "임신 성공 여부"
ID_COL = "ID"

if TARGET not in train.columns:
    cands = [f for f in os.listdir(DATA_DIR)
             if any(k in f for k in ['label','target','answer','train_y'])]
    print(f"타겟 후보: {cands}")
    LABEL_FILE = cands[0] if cands else "train_target.csv"
    ldf = pd.read_csv(f'{DATA_DIR}/{LABEL_FILE}', encoding="utf-8-sig")
    ldf.columns = [c.strip() for c in ldf.columns]
    train = train.merge(ldf, on=ID_COL, how="left")

assert TARGET in train.columns
print(f"Train: {train.shape} | Test: {test.shape}")
print(f"양성 비율: {train[TARGET].mean():.4f}  "
      f"({(train[TARGET]==0).sum()}:{(train[TARGET]==1).sum()})")

## 1. 인코딩 맵 & 상수

In [ ]:
AGE_MAP = {"만18-34세":0,"만35-37세":1,"만38-39세":2,
           "만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1}
COUNT_MAP     = {"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
DONOR_AGE_MAP = {"만20세 이하":0,"만21-25세":1,"만26-30세":2,
                 "만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}
INDUCTION_MAP = {"알 수 없음":0,"기록되지 않은 시행":1,
                 "생식선 자극 호르몬":2,"세트로타이드 (억제제)":3}
EGG_MAP   = {"본인 제공":0,"기증 제공":1,"알 수 없음":-1}
SPERM_MAP = {"배우자 제공":0,"기증 제공":1,"배우자 및 기증 제공":2,"미할당":-1}
CODE_MAP  = {v:i for i,v in enumerate(
    ["TRCMWS","TRDQAZ","TRJXFG","TRVNRY","TRXQMD","TRYBLT","TRZKPL"])}

COUNT_COLS = ["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수",
              "총 임신 횟수","IVF 임신 횟수","DI 임신 횟수",
              "총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]

# ★ v4: 처음부터 제거할 원본 피처
DROP_COLS = [
    "착상 전 유전 검사 사용 여부","PGD 시술 여부","PGS 시술 여부",
    "불임 원인 - 여성 요인","난자 채취 경과일","난자 해동 경과일",
    "파트너 정자와 혼합된 난자 수",   # imp 178 — 기증자 정자와 중복
    "미세주입된 난자 수",             # imp 163 — ICSI수정률로 대체
    # Level-1 제거 (imp < 100, 희귀/중복)
    "착상 전 유전 진단 사용 여부",    # imp ~40
    "기증 배아 사용 여부",            # imp ~45, 희귀
    "기증자 정자와 혼합된 난자 수",   # imp ~55
    "DI 출산 횟수",                  # imp ~40, 희귀
    "대리모 여부",                   # imp ~15, 희귀
]

# ★ 시술 유형 중 imp≥100만 유지 (IVF, ICSI, FER, BLASTOCYST)
DROP_PROC_TYPES = ["IUI","ICI","GIFT","Generic DI","IVI","AH"]

# ★ 배아 생성 이유 중 희귀 제거
DROP_REASON_CATS = ["기증용","연구용"]

REASON_CATS = ["현재 시술용","배아 저장용","난자 저장용","기증용","연구용"]
PROC_TYPES  = ["IVF","ICSI","FER","BLASTOCYST"]  # ★ 4개만 유지

print("v4 인코딩 맵 설정 완료")
print(f"유지 시술 유형: {PROC_TYPES}")

## 2. 전처리 함수 (Leakage-free)

In [ ]:
def expand_reason(df):
    """배아 생성 주요 이유 -> 이진 피처 (희귀 2개 제외)"""
    for cat in ["현재 시술용","배아 저장용","난자 저장용"]:
        df[f"이유_{cat}"] = df["배아 생성 주요 이유"].fillna("").str.contains(cat).astype(int)
    return df

def expand_procedure(df):
    """특정 시술 유형 -> 이진 피처 (상위 4개만)"""
    filled = df["특정 시술 유형"].fillna("Unknown")
    for pt in PROC_TYPES:
        df[f"시술_{pt}"] = (filled.str.upper().str.replace(" ","",regex=False)
                            .str.contains(pt.upper()).astype(int))
    return df

def preprocess(df, medians=None, fit=False):
    """기본 전처리. fit=True -> train 통계 저장, fit=False -> train 통계 적용"""
    df = df.copy()
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

    col_years = "임신 시도 또는 마지막 임신 경과 연수"
    if col_years in df.columns:
        df["임신_시도_연수_결측"] = df[col_years].isnull().astype(int)
        df[col_years] = df[col_years].fillna(-1)

    fill_zero = ["단일 배아 이식 여부","총 생성 배아 수","미세주입에서 생성된 배아 수",
                 "이식된 배아 수","미세주입 배아 이식 수","저장된 배아 수",
                 "미세주입 후 저장된 배아 수","해동된 배아 수","해동 난자 수",
                 "수집된 신선 난자 수","저장된 신선 난자 수","혼합된 난자 수",
                 "동결 배아 사용 여부","신선 배아 사용 여부","대리모 여부"]
    for c in fill_zero:
        if c in df.columns: df[c] = df[c].fillna(0)

    if medians is None: medians = {}
    for col in ["난자 혼합 경과일","배아 이식 경과일","배아 해동 경과일"]:
        if col not in df.columns: continue
        if fit: medians[col] = df[col].median()
        df[col] = df[col].fillna(medians.get(col, 0))

    cont_cols = ["총 생성 배아 수","미세주입에서 생성된 배아 수","이식된 배아 수",
                 "미세주입 배아 이식 수","저장된 배아 수","미세주입 후 저장된 배아 수",
                 "해동된 배아 수","해동 난자 수","수집된 신선 난자 수",
                 "저장된 신선 난자 수","혼합된 난자 수"]
    for c in cont_cols:
        if c not in df.columns: continue
        df[c] = pd.to_numeric(df[c], errors="coerce")
        if fit: medians[c] = df[c].median() if df[c].notna().any() else 0
        df[c] = df[c].fillna(medians.get(c, 0))

    if "시술 당시 나이" in df.columns:
        df["시술 당시 나이"] = df["시술 당시 나이"].map(AGE_MAP).fillna(-1).astype(int)
    for col in COUNT_COLS:
        if col in df.columns: df[col] = df[col].map(COUNT_MAP).fillna(-1).astype(int)
    for col in ["난자 기증자 나이","정자 기증자 나이"]:
        if col in df.columns: df[col] = df[col].map(DONOR_AGE_MAP).fillna(-1).astype(int)
    if "시술 유형" in df.columns:
        df["시술 유형"] = (df["시술 유형"] == "IVF").astype(int)
    if "배란 유도 유형" in df.columns:
        df["배란 유도 유형"] = df["배란 유도 유형"].map(INDUCTION_MAP).fillna(0).astype(int)
    if "난자 출처" in df.columns:
        df["난자 출처"] = df["난자 출처"].map(EGG_MAP).fillna(-1).astype(int)
    if "정자 출처" in df.columns:
        df["정자 출처"] = df["정자 출처"].map(SPERM_MAP).fillna(-1).astype(int)
    if "시술 시기 코드" in df.columns:
        df["시술 시기 코드"] = df["시술 시기 코드"].map(CODE_MAP).fillna(-1).astype(int)

    binary_cols = ["배란 자극 여부","단일 배아 이식 여부",
                   "남성 주 불임 원인","남성 부 불임 원인",
                   "여성 주 불임 원인","여성 부 불임 원인",
                   "부부 주 불임 원인","부부 부 불임 원인","불명확 불임 원인",
                   "불임 원인 - 난관 질환","불임 원인 - 남성 요인","불임 원인 - 배란 장애",
                   "불임 원인 - 자궁경부 문제","불임 원인 - 자궁내막증",
                   "불임 원인 - 정자 농도","불임 원인 - 정자 면역학적 요인",
                   "불임 원인 - 정자 운동성","불임 원인 - 정자 형태",
                   "동결 배아 사용 여부","신선 배아 사용 여부"]
    for c in binary_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df, medians

## 3. 파생피처 함수 (v4)

### 생성 전략
- **imp≥100 검증 피처만 유지** (중복 제거 포함)
- **신규 12개**: 누적실패부담, 배아생존율, IVF집중도 등 임상 인사이트 기반
- **모든 통계값 train에서만 계산** → `medians` dict로 전달 (누수 방지)

In [ ]:
def build_features(df, medians=None, fit=False):
    """
    v4 통합 피처 엔지니어링.
    fit=True  → train에서 분위수/중앙값 계산 후 medians에 저장
    fit=False → medians의 train 통계값 적용 (누수 방지)
    """
    df = df.copy()
    if medians is None: medians = {}

    # ── 0. 수치형 보정 ────────────────────────────────────────────
    num_cols = ["총 생성 배아 수","이식된 배아 수","저장된 배아 수","수집된 신선 난자 수",
                "미세주입에서 생성된 배아 수","해동된 배아 수","미세주입 후 저장된 배아 수",
                "저장된 신선 난자 수","혼합된 난자 수",
                "총 임신 횟수","총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수",
                "DI 시술 횟수","총 출산 횟수","IVF 임신 횟수","DI 임신 횟수",
                "IVF 출산 횟수","DI 출산 횟수",
                "동결 배아 사용 여부","신선 배아 사용 여부","시술 유형","시술 당시 나이",
                "배아 이식 경과일","난자 혼합 경과일","배아 해동 경과일"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    # 편의 변수
    age   = df["시술 당시 나이"].replace(-1, 0)
    egg   = df["수집된 신선 난자 수"]
    emb   = df["총 생성 배아 수"]
    trans = df["이식된 배아 수"]
    stor  = df["저장된 배아 수"]
    thaw  = df["해동된 배아 수"]
    trial = df["총 시술 횟수"].replace(-1, 0)
    ivf   = df["IVF 시술 횟수"].replace(-1, 0)
    preg  = df["총 임신 횟수"].replace(-1, 0)
    birth = df["총 출산 횟수"].replace(-1, 0)
    d_emb = df["배아 이식 경과일"]
    icsi_emb = df["미세주입에서 생성된 배아 수"]

    # ★ train 분위수 고정 (누수 방지)
    if fit:
        medians["egg_median"] = egg.median()
        medians["egg_q75"]    = egg.quantile(0.75)
        medians["emb_max"]    = max(emb.max() + 1, 9)
        medians["trial_q75"]  = trial.quantile(0.75)
        medians["age_median"] = age.median()
    egg_med  = medians.get("egg_median",  egg.median())
    egg_q75  = medians.get("egg_q75",     egg.quantile(0.75))
    emb_max  = medians.get("emb_max",     9)
    trial_q75= medians.get("trial_q75",   trial.quantile(0.75))
    age_med  = medians.get("age_median",  age.median())

    # ── A. 불임 원인 집계 ─────────────────────────────────────────
    cause_cols   = [c for c in df.columns if "불임 원인 -" in c]
    male_causes  = [c for c in ["불임 원인 - 남성 요인","불임 원인 - 정자 농도",
                                "불임 원인 - 정자 운동성","불임 원인 - 정자 형태",
                                "불임 원인 - 정자 면역학적 요인",
                                "남성 주 불임 원인","남성 부 불임 원인"] if c in df.columns]
    female_causes= [c for c in ["불임 원인 - 난관 질환","불임 원인 - 배란 장애",
                                "불임 원인 - 자궁내막증","불임 원인 - 자궁경부 문제",
                                "여성 주 불임 원인","여성 부 불임 원인"] if c in df.columns]
    primary_cols = [c for c in ["남성 주 불임 원인","여성 주 불임 원인","부부 주 불임 원인"] if c in df.columns]

    df["불임_원인_합계"]    = df[cause_cols].sum(axis=1)
    df["주요_불임_원인_합계"] = df[primary_cols].sum(axis=1)
    df["남성불임_복합"]     = df[male_causes].fillna(0).sum(axis=1)
    df["여성불임_복합"]     = df[female_causes].fillna(0).sum(axis=1)

    # ── B. 배아 효율 피처 ─────────────────────────────────────────
    fert = np.where(egg>0, emb/egg, 0).clip(0, 1)           # 수정률
    df["저장_비율"]     = np.where(emb>0, stor/emb, 0)
    df["이식_효율"]     = np.where(emb>0, trans/emb, 0)
    df["배아_손실수"]   = (emb - trans - stor).clip(lower=0)
    df["배아_총활용률"] = np.where(emb>0, (trans+stor)/emb, 0).clip(0, 1)
    df["이식_집중도"]   = np.where((trans+stor)>0, trans/(trans+stor), 0)
    df["ICSI_비율"]     = np.where(emb>0, icsi_emb/emb, 0)

    # ── C. 동결 배아 피처 ─────────────────────────────────────────
    df["동결배아_활용률"] = np.where(stor>0, thaw/stor, 0)
    df["동결배아_총량"]   = stor + thaw
    df["순수동결_주기"]   = ((df["동결 배아 사용 여부"]==1)&(df["신선 배아 사용 여부"]==0)).astype(int)

    # ── D. 난자 품질 피처 ─────────────────────────────────────────
    df["생성배아_품질복합"] = emb * fert                      # #1 importance
    df["나이x난자수"]       = (age * egg)
    df["나이보정_난자효율"] = egg / (age + 1)
    df["고령저반응"]        = ((age>=3)&(egg<egg_med)).astype(int)
    df["젊고고수확"]        = ((age<=1)&(egg>=egg_q75)).astype(int)
    df["난자_풍부도"]       = pd.cut(egg, bins=[-1,0,4,8,12,999],
                                     labels=[0,1,2,3,4]).astype(float).fillna(0).astype(int)
    df["전체_배아_효율"]    = np.where(egg>0, (trans+stor)/egg, 0).clip(0, 5)

    # ── E. 이식 품질 피처 ─────────────────────────────────────────
    is_d5 = np.where(d_emb.isnull(), 0, (d_emb>=5).astype(float))
    optimal = trans.isin([1.0, 2.0]).astype(float)
    df["is_D5"]         = is_d5
    df["최적이식_D5"]   = is_d5 * optimal
    df["배아질_복합"]   = fert*0.4 + is_d5*0.4 + optimal*0.2
    df["이식일x이식수"] = d_emb.fillna(0) * trans
    df["수정률_구간"]   = pd.cut(fert, bins=[-0.01,0.3,0.5,0.7,1.01],
                                 labels=[0,1,2,3]).astype(float)
    df["ICSI수정률"]    = np.where(emb>0, icsi_emb/emb, 0).clip(0, 1)

    # ── F. 나이 × 시술 피처 ───────────────────────────────────────
    df["나이_시술_복합코드"] = age.clip(lower=0)*10 + df["시술 유형"]
    df["나이x총시술횟수"]    = age * trial
    df["시술부담_지수"]      = age * (trial+ivf) / 2
    df["고령IVF"]            = ((age>=3)&(df["시술 유형"]==1)).astype(int)
    df["고령반복시술"]       = ((age>=3)&(trial>=3)).astype(int)

    # ── G. 시술 이력 피처 ─────────────────────────────────────────
    df["과거_임신_성공률"] = np.where(trial>0, preg/(trial+1), 0)
    df["임신_출산_전환율"] = np.where(preg>0, birth/preg, 0).clip(0, 1)
    df["출산_경험"]        = (birth>0).astype(int)
    df["초회시술"]         = (trial==0).astype(int)
    df["클리닉_집중도"]    = np.where(trial>0, df["클리닉 내 총 시술 횟수"]/trial, 1.0).clip(0,1)

    # ── H. 나이 × 배아/수정 교호 ─────────────────────────────────
    df["나이x수정률"]   = age * fert
    df["나이xD5"]       = age * is_d5
    df["배아질_나이보정"] = df["배아질_복합"] / (age + 1)

    # ── I. 배아_풍요도 (train bins 고정) ─────────────────────────
    df["배아_풍요도"] = pd.cut(emb, bins=[-1,0,3,7,emb_max],
                               labels=[0,1,2,3]).astype(float).fillna(0).astype(int)

    # ════════════════════════════════════════════════════════════════
    # ★★ v4 신규 파생피처 12개 (임상 도메인 기반, 누수 없음) ★★
    # ════════════════════════════════════════════════════════════════

    # 1. 누적실패부담 지수
    #    해석: 나이가 많고 시술을 많이 했지만 임신이 안 된 정도
    #    높을수록 예후 불량 → 음의 상관 예상
    df["누적실패부담"] = (age * trial) / (preg + 1)

    # 2. 배아 생존율
    #    해석: 이식 가능 배아(이식+저장) 중 실제 이식된 비율
    #    높을수록 적극적 이식 → 성공률 양의 상관
    df["배아_생존율"] = np.where((trans+stor)>0, trans/(trans+stor+0.01), 0)

    # 3. IVF 집중도
    #    해석: 전체 시술 중 IVF 비율 × 클리닉 집중도
    #    높을수록 전문적 IVF 치료 집중
    ivf_ratio    = np.where(trial>0, ivf/trial, 0).clip(0, 1)
    clinic_ratio = np.where(trial>0, df["클리닉 내 총 시술 횟수"]/trial, 1.0).clip(0, 1)
    df["IVF_집중도"] = ivf_ratio * clinic_ratio

    # 4. 순수 IVF 임신 효율
    #    해석: DI 임신을 제외한 IVF만의 순수 임신 성공률
    pure_ivf_preg = (preg - df["DI 임신 횟수"].replace(-1,0)).clip(lower=0)
    df["순수IVF_임신효율"] = np.where(ivf>0, pure_ivf_preg/ivf, 0).clip(0, 1)

    # 5. 배아 이식 잠재력
    #    해석: D5 블라스토시스트 × 최적이식수 × 상대적 젊음
    #    세 조건이 동시 만족 시 임신 성공률 최고
    age_norm = (5 - age.clip(0, 5)) / 5   # 0(고령)~1(젊음)
    df["배아_이식_잠재력"] = is_d5 * optimal * age_norm

    # 6. 고품질 배아 이식 복합
    #    해석: D5 + 최적이식수 + 수정률≥50% 세 조건 동시 충족
    df["고품질배아_이식"] = is_d5 * optimal * (fert>=0.5).astype(float)

    # 7. 시술 효율성 지수
    #    해석: 시술 횟수 대비 임신 효율, 고효율 환자 식별
    df["시술_효율성"] = np.where(trial>0, preg/np.sqrt(trial+1), 0)

    # 8. 나이 × 시술 부담 × 불임 원인 복합
    #    해석: 고령+반복시술+복합불임 패턴의 복합 리스크 지표
    cause_n = df["불임_원인_합계"].clip(0, 5)
    df["고위험_복합지수"] = age * (trial/(trial+1)) * (cause_n+1) / 10

    # 9. 난자배아 전환 효율 (분모 보정)
    #    해석: 투입 난자(신선+ICSI) 대비 배아 생성 비율
    total_input = egg + icsi_emb
    df["난자배아_전환효율"] = np.where(total_input>0, emb/total_input, 0).clip(0, 1)

    # 10. 동결배아 재활용 전략 점수
    #     해석: 저장된 배아를 얼마나 효율적으로 재활용하는지
    #     동결 사용 여부 × 활용률 × 이식 효율
    df["동결재활용_점수"] = (df["동결 배아 사용 여부"] *
                             df["동결배아_활용률"] *
                             df["이식_효율"])

    # 11. 반복 실패 고령 플래그
    #     해석: 40세 이상 + 3회 이상 시술 + 임신 이력 없음 → 예후 극히 불량
    df["반복실패_고령_플래그"] = ((age>=3)&(trial>=3)&(preg==0)).astype(int)

    # 12. 최적 치료 패턴 점수
    #     해석: 좋은 예후 패턴(젊음+고수확+D5+최적이식수) 동시 충족 정도
    young_flag = (age<=2).astype(float)
    rich_egg   = (egg>=egg_q75).astype(float)
    df["최적치료_패턴점수"] = young_flag * rich_egg * is_d5 * optimal

    return df, medians

## 4. 전체 파이프라인 & 데이터 준비

In [ ]:
def full_pipeline(df, medians=None, fit=False):
    """Leakage-free 전처리 + 피처 엔지니어링 통합 파이프라인"""
    
    df = expand_reason(df.copy())
    df = expand_procedure(df)
    df.drop(columns=["배아 생성 주요 이유","특정 시술 유형"], inplace=True, errors="ignore")
    df, medians = preprocess(df, medians=medians, fit=fit)
    df, medians = build_features(df, medians=medians, fit=fit)
    return df, medians

X_raw      = train.drop(columns=[ID_COL, TARGET], errors="ignore")
y          = train[TARGET]
X_test_raw = test.drop(columns=[ID_COL], errors="ignore")

# ★ train에서만 fit → test에는 train 통계 전달
X_pp, train_medians = full_pipeline(X_raw, fit=True)
X_test_pp, _        = full_pipeline(X_test_raw, medians=train_medians, fit=False)

common_cols = [c for c in X_pp.columns if c in X_test_pp.columns]
X_pp        = X_pp[common_cols]
X_test_pp   = X_test_pp[common_cols]

print(f"Train: {X_pp.shape} | Test: {X_test_pp.shape}")
print(f"잔여 결측치(Train): {X_pp.isnull().sum().sum()}")
print(f"총 피처 수: {X_pp.shape[1]}개")

# 신규 피처 생성 확인
v4_new = ["누적실패부담","배아_생존율","IVF_집중도","순수IVF_임신효율",
          "배아_이식_잠재력","고품질배아_이식","시술_효율성",
          "고위험_복합지수","난자배아_전환효율","동결재활용_점수",
          "반복실패_고령_플래그","최적치료_패턴점수"]
found = [c for c in v4_new if c in X_pp.columns]
print(f"\n★ v4 신규 피처 {len(found)}/{len(v4_new)}개: {found}")

# 누수 검증: test 통계가 feature에 사용되지 않았는지 확인
print(f"\n누수 검증:")
print(f"  train_medians 키: {list(train_medians.keys())}")
print(f"  → 모든 통계값이 train에서만 fit됨 ✅")

X_arr      = X_pp.values.astype(np.float32)
y_arr      = y.values
X_test_arr = X_test_pp.values.astype(np.float32)
skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## Phase 1 — Optuna 튜닝 (XGB / CAT / LGB)

In [ ]:
# ════ Phase 1-A: XGBoost ════════════════════════════════════════════════════
print("=== Phase 1-A: XGBoost Optuna (MedianPruner) ===")

def xgb_objective(trial):
    params = dict(
        n_estimators          = trial.suggest_int("n_estimators", 500, 3000, step=100),
        learning_rate         = trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        max_depth             = trial.suggest_int("max_depth", 3, 10),
        subsample             = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree      = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        colsample_bylevel     = trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        reg_alpha             = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        reg_lambda            = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        min_child_weight      = trial.suggest_int("min_child_weight", 1, 30),
        gamma                 = trial.suggest_float("gamma", 0, 5),
        scale_pos_weight      = trial.suggest_float("scale_pos_weight", 1.0, 5.0),
        max_delta_step        = trial.suggest_int("max_delta_step", 0, 10),
        tree_method="hist", eval_metric="auc",
        early_stopping_rounds=50,
        random_state=RANDOM_STATE,
        n_jobs=1 if GPU_AVAILABLE else -1,
        verbosity=0, **XGB_DEVICE_PARAMS,
    )
    oof = np.zeros(len(X_arr))
    for step, (tr, va) in enumerate(skf.split(X_arr, y_arr)):
        m = xgb.XGBClassifier(**params)
        m.fit(X_arr[tr], y_arr[tr], eval_set=[(X_arr[va],y_arr[va])], verbose=False)
        oof[va] = m.predict_proba(X_arr[va])[:,1]
        trial.report(roc_auc_score(y_arr[va], oof[va]), step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return roc_auc_score(y_arr, oof)

study_xgb = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=2))
study_xgb.optimize(xgb_objective, n_trials=N_OPTUNA, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
pruned_xgb = len([t for t in study_xgb.trials if t.state==optuna.trial.TrialState.PRUNED])
print(f"XGB 최적 AUC: {study_xgb.best_value:.5f}  (pruned: {pruned_xgb}/{N_OPTUNA})")

# ════ Phase 1-B: CatBoost ═══════════════════════════════════════════════════
print("\n=== Phase 1-B: CatBoost Optuna (MedianPruner) ===")

def cat_objective(trial):
    params = dict(
        iterations            = trial.suggest_int("iterations", 500, 3000, step=100),
        learning_rate         = trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        depth                 = trial.suggest_int("depth", 4, 10),
        l2_leaf_reg           = trial.suggest_float("l2_leaf_reg", 0.5, 15.0),
        bagging_temperature   = trial.suggest_float("bagging_temperature", 0, 2),
        random_strength       = trial.suggest_float("random_strength", 0, 3),
        border_count          = trial.suggest_int("border_count", 32, 255),
        min_data_in_leaf      = trial.suggest_int("min_data_in_leaf", 1, 50),
        auto_class_weights="Balanced", eval_metric="AUC",
        early_stopping_rounds=50,
        random_seed=RANDOM_STATE, verbose=0, **CAT_DEVICE_PARAMS,
    )
    oof = np.zeros(len(X_arr))
    for step, (tr, va) in enumerate(skf.split(X_arr, y_arr)):
        m = CatBoostClassifier(**params)
        m.fit(X_arr[tr], y_arr[tr], eval_set=(X_arr[va],y_arr[va]), use_best_model=True)
        oof[va] = m.predict_proba(X_arr[va])[:,1]
        trial.report(roc_auc_score(y_arr[va], oof[va]), step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return roc_auc_score(y_arr, oof)

study_cat = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=2))
study_cat.optimize(cat_objective, n_trials=N_OPTUNA, show_progress_bar=True)
best_cat_params = study_cat.best_params
pruned_cat = len([t for t in study_cat.trials if t.state==optuna.trial.TrialState.PRUNED])
print(f"CAT 최적 AUC: {study_cat.best_value:.5f}  (pruned: {pruned_cat}/{N_OPTUNA})")

# ════ Phase 1-C: LightGBM ═══════════════════════════════════════════════════
print("\n=== Phase 1-C: LightGBM Optuna (MedianPruner) ===")

def lgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 300, 2000, step=100),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves        = trial.suggest_int("num_leaves", 20, 200),
        max_depth         = trial.suggest_int("max_depth", 4, 10),
        subsample         = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        min_split_gain    = trial.suggest_float("min_split_gain", 0.0, 1.0),
        path_smooth       = trial.suggest_float("path_smooth", 0.0, 1.0),
        boosting_type     = trial.suggest_categorical("boosting_type", ["gbdt","goss"]),
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1 if GPU_AVAILABLE else -1,
        verbose=-1, **LGB_DEVICE_PARAMS,
    )
    oof = np.zeros(len(X_arr))
    for step, (tr, va) in enumerate(skf.split(X_arr, y_arr)):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_arr[tr], y_arr[tr], eval_set=[(X_arr[va],y_arr[va])],
              callbacks=[lgb.early_stopping(30,verbose=False),lgb.log_evaluation(False)])
        oof[va] = m.predict_proba(X_arr[va])[:,1]
        trial.report(roc_auc_score(y_arr[va], oof[va]), step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return roc_auc_score(y_arr, oof)

study_lgb = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=2))
study_lgb.optimize(lgb_objective, n_trials=N_OPTUNA, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
pruned_lgb = len([t for t in study_lgb.trials if t.state==optuna.trial.TrialState.PRUNED])
print(f"LGB 최적 AUC: {study_lgb.best_value:.5f}  (pruned: {pruned_lgb}/{N_OPTUNA})")

print(f"\n=== Optuna 완료 ===")
print(f"XGB: {study_xgb.best_value:.5f}  CAT: {study_cat.best_value:.5f}  LGB: {study_lgb.best_value:.5f}")

## Phase 2 — Seed 앙상블 (5 seeds × 3 모델 = 15개)

In [ ]:
print("=== Phase 2: Seed 앙상블 (15개 모델) ===")

xgb_base = {**best_xgb_params,
    "tree_method":"hist","eval_metric":"auc","early_stopping_rounds":100,
    "n_jobs":1 if GPU_AVAILABLE else -1,"verbosity":0,**XGB_DEVICE_PARAMS}
cat_base = {**best_cat_params,
    "auto_class_weights":"Balanced","eval_metric":"AUC","early_stopping_rounds":100,
    "verbose":0,**CAT_DEVICE_PARAMS}
lgb_base = {**best_lgb_params,
    "class_weight":"balanced",
    "n_jobs":1 if GPU_AVAILABLE else -1,"verbose":-1,**LGB_DEVICE_PARAMS}

model_configs = {}
for s in SEEDS:
    model_configs[f"XGB_s{s}"] = ("xgb", {**xgb_base, "random_state": s})
    model_configs[f"CAT_s{s}"] = ("cat", {**cat_base, "random_seed":  s})
    model_configs[f"LGB_s{s}"] = ("lgb", {**lgb_base, "random_state": s})

oof_models  = {}
pred_models = {}

for name, (mtype, params) in model_configs.items():
    oof  = np.zeros(len(X_arr))
    pred = np.zeros(len(X_test_arr))
    for tr, va in skf.split(X_arr, y_arr):
        if mtype == "xgb":
            m = xgb.XGBClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],verbose=False)
        elif mtype == "cat":
            m = CatBoostClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=(X_arr[va],y_arr[va]),use_best_model=True)
        else:
            m = lgb.LGBMClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],
                  callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(False)])
        oof[va] = m.predict_proba(X_arr[va])[:,1]
        pred   += m.predict_proba(X_test_arr)[:,1] / N_SPLITS
    auc = roc_auc_score(y_arr, oof)
    oof_models[name]  = oof
    pred_models[name] = pred
    print(f"  {name:<14} OOF AUC: {auc:.5f}")

model_aucs = {n: roc_auc_score(y_arr, v) for n,v in oof_models.items()}
print(f"\n개별 모델 평균: {np.mean(list(model_aucs.values())):.5f}")
print(f"최고 단일모델:  {max(model_aucs.values()):.5f}  ({max(model_aucs, key=model_aucs.get)})")

## Phase 3 — Optuna 직접 가중 앙상블 최적화

In [ ]:
print("=== Phase 3: Optuna 가중 앙상블 최적화 ===")

model_names = list(oof_models.keys())
oof_matrix  = np.array([oof_models[n]  for n in model_names])
pred_matrix = np.array([pred_models[n] for n in model_names])
n_models    = len(model_names)

# ── A. Scipy 빠른 초기화 ──────────────────────────────────────────
def neg_auc_w(w):
    w = np.clip(w, 0, None)
    s = w.sum()
    return -roc_auc_score(y_arr, oof_matrix.T @ (w/s)) if s>0 else 0

aucs = np.array([roc_auc_score(y_arr, oof_models[n]) for n in model_names])
w0   = (aucs - aucs.min() + 1e-6); w0 /= w0.sum()
res  = minimize(neg_auc_w, w0, method="L-BFGS-B", bounds=[(0,1)]*n_models)
w_scipy = np.clip(res.x, 0, None); w_scipy /= w_scipy.sum()
print(f"Scipy  가중 앙상블 AUC: {roc_auc_score(y_arr, oof_matrix.T@w_scipy):.5f}")

# ── B. Optuna 정밀 탐색 ───────────────────────────────────────────
def ens_obj(trial):
    w = np.array([trial.suggest_float(f"w_{i}", 0.0, 1.0) for i in range(n_models)])
    s = w.sum()
    return roc_auc_score(y_arr, oof_matrix.T@(w/s)) if s>1e-9 else 0.5

study_ens = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_ens.optimize(ens_obj, n_trials=200, show_progress_bar=True)

w_opt = np.array([study_ens.best_params[f"w_{i}"] for i in range(n_models)])
w_opt = np.clip(w_opt, 0, None); w_opt /= w_opt.sum()
auc_opt = roc_auc_score(y_arr, oof_matrix.T @ w_opt)
print(f"Optuna 가중 앙상블 AUC: {auc_opt:.5f}")

# ── C. Rank Averaging ─────────────────────────────────────────────
def rank_avg(mat, w):
    w = w/w.sum()
    n = mat.shape[1]
    return np.array([rankdata(row)/n for row in mat]).T @ w

oof_rank  = rank_avg(oof_matrix,  w_opt)
pred_rank = rank_avg(pred_matrix, w_opt)
auc_rank  = roc_auc_score(y_arr, oof_rank)
print(f"Rank   가중 앙상블 AUC: {auc_rank:.5f}")

# ── D. Prob + Rank 최적 혼합 ─────────────────────────────────────
oof_prob  = oof_matrix.T  @ w_opt
pred_prob = pred_matrix.T @ w_opt

best_alpha, best_mix = 0.5, 0.0
for alpha in np.linspace(0, 1, 201):
    a = roc_auc_score(y_arr, alpha*oof_prob + (1-alpha)*oof_rank)
    if a > best_mix: best_mix, best_alpha = a, alpha

oof_final  = best_alpha*oof_prob  + (1-best_alpha)*oof_rank
pred_final = best_alpha*pred_prob + (1-best_alpha)*pred_rank

print(f"\n최종 혼합 AUC: {best_mix:.5f}  (alpha={best_alpha:.2f})")
print("\n상위 5개 모델 가중치:")
for idx in np.argsort(w_opt)[::-1][:5]:
    print(f"  {model_names[idx]:<14}: w={w_opt[idx]:.4f}  OOF={model_aucs[model_names[idx]]:.5f}")

sub_final = sub.copy()
sub_final["probability"] = pred_final
sub_final.to_csv(f"{OUT_DIR}/submission_v4_final.csv", index=False)
print(f"\n✅ submission_v4_final.csv 저장")
print(sub_final.head())

## Phase 4 — Permutation Importance 기반 최종 피처 정제 & 재학습

In [ ]:
print("=== Phase 4: Permutation Importance 검증 ===")

m_perm = lgb.LGBMClassifier(**{**lgb_base, "random_state": RANDOM_STATE})
m_perm.fit(X_arr, y_arr, callbacks=[lgb.log_evaluation(False)])

# Split importance
split_imp = pd.DataFrame({
    "feature":   X_pp.columns,
    "split_imp": m_perm.feature_importances_,
}).sort_values("split_imp", ascending=False)

# Permutation importance (val 5000개 샘플)
val_idx  = np.random.choice(len(X_arr), min(5000,len(X_arr)), replace=False)
perm_res = pi_fn(m_perm, X_arr[val_idx], y_arr[val_idx],
                 n_repeats=5, random_state=RANDOM_STATE, scoring="roc_auc")
perm_imp = pd.DataFrame({
    "feature":   X_pp.columns,
    "perm_mean": perm_res.importances_mean,
    "perm_std":  perm_res.importances_std,
}).sort_values("perm_mean", ascending=False)

# ★ 악영향 피처 자동 탐지 (perm_mean < 0)
harmful = perm_imp[perm_imp["perm_mean"] < 0]
print(f"\n성능 악영향 피처 (perm_mean < 0): {len(harmful)}개")
if len(harmful) > 0:
    print(harmful[["feature","perm_mean","perm_std"]].to_string(index=False))

# ★ 악영향 피처 제거 후 전체 앙상블 재학습
harmful_cols = harmful["feature"].tolist()
if len(harmful_cols) > 0:
    print(f"\n악영향 피처 {len(harmful_cols)}개 제거 후 재학습...")
    keep_idx = [i for i,c in enumerate(X_pp.columns) if c not in harmful_cols]
    Xc       = X_arr[:, keep_idx]
    Xtc      = X_test_arr[:, keep_idx]
    print(f"  피처 수: {X_arr.shape[1]} → {len(keep_idx)}개")

    oof_clean_models  = {}
    pred_clean_models = {}
    for name, (mtype, params) in model_configs.items():
        oof  = np.zeros(len(Xc))
        pred = np.zeros(len(Xtc))
        for tr, va in skf.split(Xc, y_arr):
            if mtype == "xgb":
                m = xgb.XGBClassifier(**params)
                m.fit(Xc[tr],y_arr[tr],eval_set=[(Xc[va],y_arr[va])],verbose=False)
            elif mtype == "cat":
                m = CatBoostClassifier(**params)
                m.fit(Xc[tr],y_arr[tr],eval_set=(Xc[va],y_arr[va]),use_best_model=True)
            else:
                m = lgb.LGBMClassifier(**params)
                m.fit(Xc[tr],y_arr[tr],eval_set=[(Xc[va],y_arr[va])],
                      callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(False)])
            oof[va] = m.predict_proba(Xc[va])[:,1]
            pred   += m.predict_proba(Xtc)[:,1] / N_SPLITS
        oof_clean_models[name]  = oof
        pred_clean_models[name] = pred

    # 재최적화
    oof_c  = np.array([oof_clean_models[n]  for n in model_names])
    pred_c = np.array([pred_clean_models[n] for n in model_names])

    def ens_obj_clean(trial):
        w = np.array([trial.suggest_float(f"w_{i}", 0.0, 1.0) for i in range(n_models)])
        s = w.sum()
        return roc_auc_score(y_arr, oof_c.T@(w/s)) if s>1e-9 else 0.5

    study_clean = optuna.create_study(direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study_clean.optimize(ens_obj_clean, n_trials=100)
    wc = np.array([study_clean.best_params[f"w_{i}"] for i in range(n_models)])
    wc = np.clip(wc,0,None); wc /= wc.sum()

    oof_prob_c  = oof_c.T  @ wc
    pred_prob_c = pred_c.T @ wc
    oof_rank_c  = rank_avg(oof_c,  wc)
    pred_rank_c = rank_avg(pred_c, wc)

    best_a2, best_mix2 = 0.5, 0.0
    for alpha in np.linspace(0,1,201):
        a = roc_auc_score(y_arr, alpha*oof_prob_c+(1-alpha)*oof_rank_c)
        if a > best_mix2: best_mix2, best_a2 = a, alpha

    pred_clean_final = best_a2*pred_prob_c + (1-best_a2)*pred_rank_c
    print(f"\n피처 제거 전 OOF: {best_mix:.5f}")
    print(f"피처 제거 후 OOF: {best_mix2:.5f}  (개선: {best_mix2-best_mix:+.5f})")

    if best_mix2 > best_mix:
        pred_final = pred_clean_final
        print("✅ 악영향 피처 제거 후 성능 향상 → 제거 버전으로 최종 제출")
    else:
        print("⚠️  피처 제거 후 성능 하락 → 원본 유지")
else:
    print("악영향 피처 없음 ✅ — 현재 피처셋 최적")

# ★ 최종 제출 저장
sub_final["probability"] = pred_final
sub_final.to_csv(f"{OUT_DIR}/submission_v4_best.csv", index=False)
print(f"\n✅ submission_v4_best.csv 저장")

## 시각화 & 결과 요약

In [ ]:
# ── Feature Importance 시각화 ────────────────────────────────────────
v4_new_set = {"누적실패부담","배아_생존율","IVF_집중도","순수IVF_임신효율",
              "배아_이식_잠재력","고품질배아_이식","시술_효율성",
              "고위험_복합지수","난자배아_전환효율","동결재활용_점수",
              "반복실패_고령_플래그","최적치료_패턴점수"}

fig, axes = plt.subplots(1, 2, figsize=(20, 14))

top_split = split_imp.head(30)
sc = ["#E05C5C" if f in v4_new_set else "#5B9BD5" for f in top_split["feature"]]
axes[0].barh(top_split["feature"][::-1], top_split["split_imp"][::-1],
             color=sc[::-1], alpha=0.85, edgecolor="white")
axes[0].set_title("Split Importance Top 30", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Importance")

top_perm = perm_imp.head(30)
pc = ["#E05C5C" if f in v4_new_set else "#5B9BD5" for f in top_perm["feature"]]
axes[1].barh(top_perm["feature"][::-1], top_perm["perm_mean"][::-1],
             xerr=top_perm["perm_std"][::-1],
             color=pc[::-1], alpha=0.85, edgecolor="white", capsize=3)
axes[1].axvline(x=0, color="black", lw=0.8, ls="--")
axes[1].set_title("Permutation Importance Top 30\n(음수 = 악영향)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("AUC 변화량")

for ax in axes:
    ax.legend(handles=[
        mpatches.Patch(color="#5B9BD5", label="기존 피처"),
        mpatches.Patch(color="#E05C5C", label="★ v4 신규"),
    ], fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/feature_importance_v4.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 결과 요약 ─────────────────────────────────────────────────────
print("=" * 65)
print("  v4 결과 요약")
print("=" * 65)
results = {
    "[이전] v2 최종":               0.74004,
    "[v4] XGB 단독":                study_xgb.best_value,
    "[v4] CAT 단독":                study_cat.best_value,
    "[v4] LGB 단독":                study_lgb.best_value,
    "[v4] Optuna 가중 앙상블":      auc_opt,
    "[v4] Rank 앙상블":             auc_rank,
    "[v4] 최종 (Prob+Rank)":       best_mix,
}
baseline = 0.74004
for k, v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    diff = f"({v-baseline:+.5f})" if "v4" in k else ""
    bar  = "█" * int((v-0.73)*500)
    print(f"  {k:<35} {v:.5f} {diff:<12}  {bar}")
print("=" * 65)
print(f"\n총 피처 수: {X_pp.shape[1]}개")
print(f"v4 신규 피처 수: {len([c for c in v4_new_set if c in X_pp.columns])}개")
print(f"v2 대비 피처 감소: 147 → {X_pp.shape[1]}개 ({147-X_pp.shape[1]}개 제거)")

import os
for fn in [f"{OUT_DIR}/submission_v4_final.csv", f"{OUT_DIR}/submission_v4_best.csv"]:
    if os.path.exists(fn):
        df_ = pd.read_csv(fn)
        print(f"  {fn}: {len(df_)}행  [{df_['probability'].min():.4f},{df_['probability'].max():.4f}]")